In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = 8.5 AND final_rating = 100 (numeric token extraction)
      - primary_product_serv matches the predefined list of 14 categories.
   3. ENGAGEMENT-GRAIN FLAGGING: Evaluate the two rule flags across all rows for the
      same EngagementNumber before attempting the AU bridge.
   4. COMMA EXCEPTION HANDLING: Protects specific unit names containing commas using
      the '[COMMA]' placeholder swap before explode().
   5. FANOUT RULE: Preserve one-to-many mapping from TPRM_Units to multiple in-scope
      master AUs. Deduplicate only at (BusinessID, EngagementNumber).
   6. FINAL OUTPUT: Binary 'Yes' / 'No' output based on whether any qualified
      engagement bridged to the AU.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        KPI_Number AS Raw_KPI_Number,
        final_rating AS Raw_Final_Rating,
        TRY_CAST(REGEXP_EXTRACT(TRIM(CAST(KPI_Number AS STRING)), '[-]?[0-9]+(\.[0-9]+)?', 0) AS DOUBLE) AS Parsed_KPI_Number,
        TRY_CAST(REGEXP_EXTRACT(TRIM(CAST(final_rating AS STRING)), '[-]?[0-9]+(\.[0-9]+)?', 0) AS DOUBLE) AS Parsed_Final_Rating,
        primary_product_serv,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Engagement_Flags AS (
    SELECT
        EngagementNumber,
        MAX(CASE WHEN Parsed_KPI_Number = 8.5 AND Parsed_Final_Rating = 100.0 THEN 1 ELSE 0 END) AS KPI_85_100_Flag,
        MAX(CASE WHEN UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        ) THEN 1 ELSE 0 END) AS Product_Service_Flag
    FROM Third_Party_Risk_Data
    GROUP BY EngagementNumber
    HAVING MAX(CASE WHEN Parsed_KPI_Number = 8.5 AND Parsed_Final_Rating = 100.0 THEN 1 ELSE 0 END) = 1
        OR MAX(CASE WHEN UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        ) THEN 1 ELSE 0 END) = 1
),

LOB_Source_By_Engagement AS (
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE COALESCE(TRIM(safe_owning_lob), '') <> ''
       OR COALESCE(TRIM(safe_lob_using), '') <> ''
),

Flagged_Engagements AS (
    SELECT DISTINCT
        f.EngagementNumber,
        f.KPI_85_100_Flag,
        f.Product_Service_Flag,
        l.safe_owning_lob,
        l.safe_lob_using
    FROM Engagement_Flags f
    INNER JOIN LOB_Source_By_Engagement l
        ON f.EngagementNumber = l.EngagementNumber
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber,
        KPI_85_100_Flag,
        Product_Service_Flag,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Resolved_Bridge AS (
    SELECT DISTINCT
        mast.BusinessID,
        f.EngagementNumber
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
),

Engagements_By_AU AS (
    SELECT 
        BusinessID,
        COUNT(DISTINCT EngagementNumber) AS Match_Count
    FROM Resolved_Bridge
    GROUP BY BusinessID
)

SELECT 
    a.BusinessID, 
    a.AU_Name, 
    a.Business_Segment,
    'TP03' AS QuestionID,
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - AU Level Calculation Review
   PURPOSE: One row per AU showing bridged engagements, bridge keys, and whether
   those engagements qualified by KPI 8.5/final_rating 100, product/service category,
   or both. This is the single consolidated debug query for TP03.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        KPI_Number AS Raw_KPI_Number,
        final_rating AS Raw_Final_Rating,
        TRY_CAST(REGEXP_EXTRACT(TRIM(CAST(KPI_Number AS STRING)), '[-]?[0-9]+(\.[0-9]+)?', 0) AS DOUBLE) AS Parsed_KPI_Number,
        TRY_CAST(REGEXP_EXTRACT(TRIM(CAST(final_rating AS STRING)), '[-]?[0-9]+(\.[0-9]+)?', 0) AS DOUBLE) AS Parsed_Final_Rating,
        primary_product_serv,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Engagement_Flags AS (
    SELECT
        EngagementNumber,
        MAX(CASE WHEN Parsed_KPI_Number = 8.5 AND Parsed_Final_Rating = 100.0 THEN 1 ELSE 0 END) AS KPI_85_100_Flag,
        MAX(CASE WHEN UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        ) THEN 1 ELSE 0 END) AS Product_Service_Flag
    FROM Third_Party_Risk_Data
    GROUP BY EngagementNumber
    HAVING MAX(CASE WHEN Parsed_KPI_Number = 8.5 AND Parsed_Final_Rating = 100.0 THEN 1 ELSE 0 END) = 1
        OR MAX(CASE WHEN UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        ) THEN 1 ELSE 0 END) = 1
),

LOB_Source_By_Engagement AS (
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE COALESCE(TRIM(safe_owning_lob), '') <> ''
       OR COALESCE(TRIM(safe_lob_using), '') <> ''
),

Flagged_Engagements AS (
    SELECT DISTINCT
        f.EngagementNumber,
        f.KPI_85_100_Flag,
        f.Product_Service_Flag,
        l.safe_owning_lob,
        l.safe_lob_using
    FROM Engagement_Flags f
    INNER JOIN LOB_Source_By_Engagement l
        ON f.EngagementNumber = l.EngagementNumber
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber,
        KPI_85_100_Flag,
        Product_Service_Flag,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Resolved_Bridge AS (
    SELECT DISTINCT
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        f.EngagementNumber,
        f.KPI_85_100_Flag,
        f.Product_Service_Flag,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
),

AU_Debug AS (
    SELECT 
        BusinessID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        COUNT(DISTINCT EngagementNumber) AS Distinct_Engagement_Count,
        COUNT(DISTINCT CASE WHEN KPI_85_100_Flag = 1 THEN EngagementNumber END) AS KPI_85_100_Engagement_Count,
        COUNT(DISTINCT CASE WHEN Product_Service_Flag = 1 THEN EngagementNumber END) AS Product_Service_Engagement_Count,
        COUNT(DISTINCT CASE WHEN KPI_85_100_Flag = 1 AND Product_Service_Flag = 1 THEN EngagementNumber END) AS Both_Flags_Engagement_Count,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN KPI_85_100_Flag = 1 THEN CAST(EngagementNumber AS STRING) END))) AS KPI_85_100_Engagement_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Product_Service_Flag = 1 THEN CAST(EngagementNumber AS STRING) END))) AS Product_Service_Engagement_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN KPI_85_100_Flag = 1 AND Product_Service_Flag = 1 THEN CAST(EngagementNumber AS STRING) END))) AS Both_Flags_Engagement_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(TPRM_Units))) AS Matched_TPRM_Units_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Mapping_ID_Or_Name))) AS Mapping_Bridge_Value_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(EngagementNumber AS STRING)))) AS Engagement_List
    FROM Resolved_Bridge
    GROUP BY BusinessID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'TP03' AS QuestionID,
    CASE WHEN COALESCE(d.Distinct_Engagement_Count, 0) > 0 THEN 'Yes' ELSE 'No' END AS Response,
    COALESCE(d.Matched_TPRM_Units_List, 'n/a') AS Matched_TPRM_Units_List,
    COALESCE(d.Mapping_Bridge_Value_List, 'n/a') AS Mapping_Bridge_Value_List,
    COALESCE(d.Engagement_List, 'n/a') AS Engagement_List,
    COALESCE(d.Distinct_Engagement_Count, 0) AS Distinct_Engagement_Count,
    COALESCE(d.KPI_85_100_Engagement_Count, 0) AS KPI_85_100_Engagement_Count,
    COALESCE(d.KPI_85_100_Engagement_List, 'n/a') AS KPI_85_100_Engagement_List,
    COALESCE(d.Product_Service_Engagement_Count, 0) AS Product_Service_Engagement_Count,
    COALESCE(d.Product_Service_Engagement_List, 'n/a') AS Product_Service_Engagement_List,
    COALESCE(d.Both_Flags_Engagement_Count, 0) AS Both_Flags_Engagement_Count,
    COALESCE(d.Both_Flags_Engagement_List, 'n/a') AS Both_Flags_Engagement_List,
    CASE 
        WHEN COALESCE(d.Distinct_Engagement_Count, 0) > 0 THEN CONCAT(
            'Yes because ', CAST(d.Distinct_Engagement_Count AS STRING), ' distinct bridged engagement(s) qualified. KPI 8.5/final_rating 100=',
            CAST(d.KPI_85_100_Engagement_Count AS STRING), ', product/service=', CAST(d.Product_Service_Engagement_Count AS STRING),
            ', both=', CAST(d.Both_Flags_Engagement_Count AS STRING)
        )
        ELSE NULL
    END AS Why_Yes,
    CASE 
        WHEN COALESCE(d.Distinct_Engagement_Count, 0) = 0 THEN 'No because no flagged engagement bridged to this AU'
        ELSE NULL
    END AS Why_No,
    'Flags are evaluated at EngagementNumber grain before bridging, so KPI and product/service evidence can both appear for the same bridged engagement.' AS Debug_Reason,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.BusinessID
ORDER BY mast.BusinessID;
